# CP1 · U13 — pregătirea și verificarea datelor

Notebookul conține codul pipeline-ului, în ordinea de rulare. Pentru o rulare completă: **Restart Kernel → Run All**, sau în terminal `.venv/bin/python ruleaza.py toate`. Pentru etape separate folosește `ruleaza.py --help`; selecția se face prin numele etapelor, nu prin poziția celulelor.

Inventarierea obișnuită citește numai metadatele. Scanarea integrală a tuturor rasterelor este opțională: `.venv/bin/python ruleaza.py inventar --scanare-integrala`. În inventarul rapid, statisticile globale necalculate sunt lăsate goale. Etapele următoare citesc doar zona de studiu.

Grilă: 100 m, EPSG:32635, celule cu centrul în București. Pragul de 80% este provizoriu. Valorile brute, NoData, QA/SCL, unitățile și limitele sunt explicate în secțiunile următoare și în `prezentare/datacard_U13.pdf`.

**Etapele au dependențe explicite:** pregătire → funcții → grilă → aliniere. Hărțile și diagnosticul citesc tabelul salvat. Nu este suficient să execuți arbitrar ultima celulă într-un context gol.


**Repo public:** https://github.com/colacbrr/Bucharest-Urban-Heat. Fișele de descărcare, identificatorii și licențele sunt în [datacard_U13.pdf](prezentare/datacard_U13.pdf). Rasterele nu sunt incluse în Git: descarcă produsele indicate în fișe și păstrează numele și structura în `date/landsat/`, `date/sentinel2/` și `date/ghsl/`. Cerințe de rulare: Python 3.14 și `pip install -r requirements.txt`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform

ROOT = Path.cwd()
assert (ROOT / 'date').is_dir(), 'Rulează din rădăcina proiectului.'
files = []

def add(source, pattern):
    matches = sorted(ROOT.glob(pattern))
    if len(matches) != 1:
        raise ValueError(f'{source}: așteptam exact un fișier pentru {pattern}; găsite {len(matches)}')
    files.append((source, matches[0]))

for band in ['ST_B10', 'QA_PIXEL', 'ST_QA']:
    add('Landsat_' + band, f'date/landsat/*_{band}.TIF')
for product, prefix in [('BUILT_H','GHS_BUILT_H_AGBH_E2018'),
                        ('BUILT_S','GHS_BUILT_S_E2020'), ('POP','GHS_POP_E2020')]:
    add('GHSL_' + product, f'date/ghsl/{prefix}_GLOBE_R2023A_54009_100_V1_0_R4_C21.tif')
for band in ['B02_10m', 'B03_10m', 'B04_10m', 'B08_10m', 'B11_20m', 'SCL_20m']:
    add('Sentinel2_' + band, f'date/sentinel2/*.SAFE/GRANULE/*/IMG_DATA/*/*_{band}.jp2')
print(f'{len(files)} rastere selectate din Landsat, GHSL BUILT-H/BUILT-S/POP și Sentinel-2.')
# Scanarea integrală este un control separat; rularea obișnuită citește metadatele.
import os
FULL_SCAN = os.environ.get('MEDE_FULL_SCAN', '0') == '1'


12 rastere selectate din Landsat, GHSL BUILT-H/BUILT-S/POP și Sentinel-2.


In [2]:
rows = []
for source, path in files:
    with rasterio.open(path) as ds:
        if ds.crs is None or ds.count != 1:
            raise ValueError(f'CRS absent sau număr de benzi neașteptat: {path.name}')
        valid = 0
        low, high = np.inf, -np.inf
        if FULL_SCAN:
            for _, window in ds.block_windows(1):
                values = ds.read(1, window=window, masked=True).compressed()
                values = values[np.isfinite(values)]
                valid += values.size
                if values.size:
                    low = min(low, float(values.min()))
                    high = max(high, float(values.max()))
        x, y = transform('EPSG:4326', ds.crs, [26.1025], [44.4268])
        row, col = ds.index(x[0], y[0])
        inside = 0 <= row < ds.height and 0 <= col < ds.width
        record = dict(sursa=source, fisier=str(path.relative_to(ROOT)),
                      crs=ds.crs.to_string(), rezolutie_x=ds.res[0], rezolutie_y=ds.res[1],
                      latime=ds.width, inaltime=ds.height, dtype=ds.dtypes[0],
                      nodata=ds.nodata, pixeli_total=ds.width*ds.height,
                      pixeli_valizi_masca_fisier=valid if FULL_SCAN else None,
                      procent_valid_masca_fisier=100*valid/(ds.width*ds.height) if FULL_SCAN else None,
                      scanare_integrala=FULL_SCAN, marime_bytes=path.stat().st_size,
                      mtime_ns=path.stat().st_mtime_ns,
                      minim_stocat=low if valid else None, maxim_stocat=high if valid else None,
                      centru_bucuresti_in_raster=inside)
        rows.append(record)
        print(f'{source}: {"scanare integrală" if FULL_SCAN else "metadate"}; {ds.crs}; centru București inclus: {inside}')
inventar = pd.DataFrame(rows)
inventar.to_csv(ROOT / 'date/inventar.csv', index=False)
print('\nInventar salvat în date/inventar.csv')
columns = ['sursa','crs','rezolutie_x','rezolutie_y','nodata']
if FULL_SCAN:
    columns += ['minim_stocat','maxim_stocat','procent_valid_masca_fisier']
else:
    print('Mod rapid: statisticile globale nu sunt calculate. Pentru ele: inventar --scanare-integrala')
print(inventar[columns].to_string(index=False))

Landsat_ST_B10: metadate; EPSG:32635; centru București inclus: True
Landsat_QA_PIXEL: metadate; EPSG:32635; centru București inclus: True
Landsat_ST_QA: metadate; EPSG:32635; centru București inclus: True
GHSL_BUILT_H: metadate; ESRI:54009; centru București inclus: True
GHSL_BUILT_S: metadate; ESRI:54009; centru București inclus: True
GHSL_POP: metadate; ESRI:54009; centru București inclus: True
Sentinel2_B02_10m: metadate; EPSG:32635; centru București inclus: True
Sentinel2_B03_10m: metadate; EPSG:32635; centru București inclus: True
Sentinel2_B04_10m: metadate; EPSG:32635; centru București inclus: True
Sentinel2_B08_10m: metadate; EPSG:32635; centru București inclus: True
Sentinel2_B11_20m: metadate; EPSG:32635; centru București inclus: True
Sentinel2_SCL_20m: metadate; EPSG:32635; centru București inclus: True

Inventar salvat în date/inventar.csv
Mod rapid: statisticile globale nu sunt calculate. Pentru ele: inventar --scanare-integrala
            sursa        crs  rezolutie_x  re

## Conturul Bucureștiului și acoperirea geometrică

Rulare: `.venv/bin/python ruleaza.py geometrie`. Folosește sursele descoperite în pregătire și numai metadatele rasterelor.

Selectăm poligonul OSM `relation/377733`, ignorând punctul auxiliar. Verificăm validitatea topologică fără reparare automată și calculăm suprafețele în **EPSG:32635 (UTM 35N)**.

Pentru fiecare raster intersectăm conturul cu amprenta dreptunghiului său, reproiectată în UTM. Rezultatul măsoară **acoperirea geometrică**, nu pixelii utilizabili: NoData, norii și QA/SCL vor fi evaluate separat. O amprentă de 100% poate include goluri în date. Raportul se salvează în `date/acoperire_geometrica.csv`.

In [3]:
from pathlib import Path
import json
import pandas as pd
import rasterio
from rasterio.warp import transform as transform_coords
from shapely.geometry import shape, Polygon
from shapely import transform as transform_shape
from shapely.validation import explain_validity

PROJECT = Path.cwd()
if not (PROJECT / 'date/bucuresti.geojson').exists():
    raise FileNotFoundError('Deschide notebookul din rădăcina proiectului MEDE.')
geojson = json.loads((PROJECT / 'date/bucuresti.geojson').read_text())
features = [f for f in geojson['features']
            if f.get('geometry', {}).get('type') in ('Polygon', 'MultiPolygon')
            and f.get('properties', {}).get('@id') == 'relation/377733']
if len(features) != 1:
    raise ValueError('Exportul trebuie să conțină exact un poligon relation/377733.')
city_wgs84 = shape(features[0]['geometry'])
if city_wgs84.is_empty or not city_wgs84.is_valid:
    raise ValueError('Contur invalid: ' + explain_validity(city_wgs84))

AREA_CRS = 'EPSG:32635'
def project_geometry(geometry, src_crs, dst_crs):
    return transform_shape(geometry,
        lambda x, y: tuple(np.asarray(v) for v in transform_coords(src_crs, dst_crs, x, y)),
        interleaved=False)

# Densificăm muchiile înainte de reproiectare pentru a urmări curbura transformării.
city_utm = project_geometry(city_wgs84.segmentize(0.001), 'EPSG:4326', AREA_CRS)
if not city_utm.is_valid or city_utm.area <= 0:
    raise ValueError('Conturul reproiectat este invalid sau are suprafață nulă.')
city_area_km2 = city_utm.area / 1_000_000
print('Contur București: valid; punctul auxiliar a fost ignorat.')
print(f'Suprafață OSM calculată în UTM 35N: {city_area_km2:.3f} km²')
print('Aceasta este suprafața geometriei descărcate, nu o valoare administrativă oficială.')

Contur București: valid; punctul auxiliar a fost ignorat.
Suprafață OSM calculată în UTM 35N: 238.887 km²
Aceasta este suprafața geometriei descărcate, nu o valoare administrativă oficială.


In [4]:
source_inventory = pd.DataFrame([{'sursa': name, 'fisier': str(path.relative_to(ROOT))} for name,path in files])
coverage_rows = []
for item in source_inventory.itertuples(index=False):
    path = PROJECT / item.fisier
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f'CRS absent: {path.name}')
        corners = [ds.transform * point for point in
                   [(0, 0), (ds.width, 0), (ds.width, ds.height), (0, ds.height)]]
        footprint = Polygon(corners)
        # Sursele curente au coordonate metrice; refuzăm alte unități până alegem pasul potrivit.
        if not ds.crs.is_projected:
            raise ValueError(f'Alege un pas de densificare pentru CRS-ul neproiectat: {ds.crs}')
        footprint_utm = project_geometry(footprint.segmentize(1000), ds.crs, AREA_CRS)
        if not footprint_utm.is_valid:
            raise ValueError(f'Amprentă reproiectată invalidă: {path.name}')
        overlap_area = city_utm.intersection(footprint_utm).area
        uncovered_area = city_utm.difference(footprint_utm).area
        coverage_rows.append({
            'sursa': item.sursa,
            'fisier': item.fisier,
            'crs_raster': ds.crs.to_string(),
            'suprafata_oras_km2': city_area_km2,
            'suprafata_acoperita_km2': overlap_area / 1_000_000,
            'suprafata_neacoperita_m2': uncovered_area,
            'acoperire_geometrica_pct': min(100.0, 100 * overlap_area / city_utm.area),
            'acopera_integral_geometric': footprint_utm.covers(city_utm),
        })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(PROJECT / 'date/acoperire_geometrica.csv', index=False)
print(coverage[['sursa', 'acoperire_geometrica_pct', 'suprafata_neacoperita_m2',
                'acopera_integral_geometric']].to_string(index=False))
print('\nSalvat: date/acoperire_geometrica.csv')
if coverage['acopera_integral_geometric'].all():
    print('Toate amprentele includ întregul contur. Urmează verificarea pixelilor valizi.')
else:
    print('Există surse cu acoperire parțială. Verificăm lipsurile înainte de aliniere.')

            sursa  acoperire_geometrica_pct  suprafata_neacoperita_m2  acopera_integral_geometric
   Landsat_ST_B10                     100.0                       0.0                        True
 Landsat_QA_PIXEL                     100.0                       0.0                        True
    Landsat_ST_QA                     100.0                       0.0                        True
     GHSL_BUILT_H                     100.0                       0.0                        True
     GHSL_BUILT_S                     100.0                       0.0                        True
         GHSL_POP                     100.0                       0.0                        True
Sentinel2_B02_10m                     100.0                       0.0                        True
Sentinel2_B03_10m                     100.0                       0.0                        True
Sentinel2_B04_10m                     100.0                       0.0                        True
Sentinel2_B08_10m   

## Pixelii utilizabili în interiorul Bucureștiului

Rulare: `.venv/bin/python ruleaza.py pixeli`. Citește doar ferestrele care cuprind orașul. Denominatorul este numărul de pixeli al căror **centru** cade în poligon, pe grila nativă a fiecărui raster; procentele nu reprezintă încă intersecția pe grila comună de 100 m.

Reguli inițiale ale proiectului:

- **Landsat ST_B10:** valoare prezentă și finită, diferită de fill 0; QA_PIXEL prezent, fără biții 0–5 (fill, dilated cloud, cirrus, cloud, cloud shadow, snow). Nu aplicăm încă un prag suplimentar ST_QA; această verificare nu certifică precizia temperaturii. [USGS — QA](https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands).
- **Sentinel-2:** bandă prezentă, finită și diferită de fill 0; păstrăm conservator clasele **SCL 4, 5, 6** (vegetație, suprafețe fără vegetație, apă). Excludem inclusiv 2 (umbre topografice/pixeli întunecați) și 7 (neclasificat), precum și NoData, defecte, umbre de nori, nori, cirrus și zăpadă. Acestea sunt alegerile noastre de filtrare, nu dovada că orice pixel exclus era nor. SCL se transferă prin **nearest neighbour**, fără interpolarea claselor. [Definițiile SCL](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html).
- **GHSL:** eliminăm numai masca NoData și valorile nefinite. **Zero rămâne valid** pentru populație, construit și înălțime.

Apa este păstrată în acest diagnostic de calitate; eventuala excludere din modelul urban va fi o decizie distinctă. Raportul separă lipsa valorilor de excluderea suplimentară prin QA/SCL, fără dublă numărare. Nu suprascriem rasterele originale.

Rezultat: `date/acoperire_pixeli.csv`, plus distribuția claselor SCL afișată în terminal.

Raportul include rezoluțiile pe ambele axe și suprafața validă aproximativă (determinantul transformării × numărul pixelilor acceptați). Centrele pixelilor definesc selecția, fără decupare vectorială la margine; în UTM, aria plană diferă ușor de aria geodezică. Biții de fill QA sunt clasificați ca lipsă, nu ca excludere suplimentară. Procentele sunt păstrate fără rotunjire în CSV.


In [5]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask, geometry_window
from rasterio.warp import transform_geom, reproject, Resampling

P = Path.cwd()
features = json.loads((P / 'date/bucuresti.geojson').read_text())['features']
selected = [f['geometry'] for f in features if f.get('properties', {}).get('@id') == 'relation/377733'
            and f['geometry']['type'] in ('Polygon', 'MultiPolygon')]
if len(selected) != 1:
    raise ValueError('Este necesar exact un poligon București.')
city = selected[0]
paths = dict(files)

def read_city(path):
    with rasterio.open(path) as ds:
        geom = transform_geom('EPSG:4326', ds.crs, city)
        window = geometry_window(ds, [geom])
        data = ds.read(1, window=window, masked=True)
        affine = ds.window_transform(window)
        inside = geometry_mask([geom], out_shape=data.shape, transform=affine,
                               invert=True, all_touched=False)
        if not inside.any():
            raise ValueError(f'Niciun centru de pixel în poligon: {path.name}')
        values = np.asarray(data.data)
        present = ~np.ma.getmaskarray(data) & np.isfinite(values)
        return values, present, inside, affine, ds.crs

from functools import lru_cache

@lru_cache(maxsize=8)
def categorical_on_grid(path, target_shape, affine, crs, fill):
    out = np.full(target_shape, fill, dtype=np.uint16)
    with rasterio.open(path) as ds:
        reproject(source=rasterio.band(ds, 1), destination=out,
                  src_transform=ds.transform, src_crs=ds.crs, src_nodata=ds.nodata,
                  dst_transform=affine, dst_crs=crs, dst_nodata=fill,
                  resampling=Resampling.nearest)
    out.setflags(write=False)
    return out

def qa_landsat_ok(qa):
    return (qa & np.uint16(63)) == 0

def scl_ok(scl):
    return np.isin(scl, [4, 5, 6])


@lru_cache(maxsize=6)
def load_native(source):
    with rasterio.open(paths[source]) as ds:
        geom = transform_geom(GRID_CRS, ds.crs, grid_bounds_geom)
        window = geometry_window(ds, [geom], pad_x=2, pad_y=2)
        data = ds.read(1, window=window, masked=True)
        affine, crs = ds.window_transform(window), ds.crs
    values = np.asarray(data.data, dtype=np.float64)
    valid = ~np.ma.getmaskarray(data) & np.isfinite(values)
    values.setflags(write=False)
    valid.setflags(write=False)
    return values, valid, affine, crs



In [6]:
SCL_NAMES = {0: 'no_data', 1: 'saturated/defective', 2: 'dark/topographic_shadow',
             3: 'cloud_shadow', 4: 'vegetation', 5: 'not_vegetated', 6: 'water',
             7: 'unclassified', 8: 'cloud_medium', 9: 'cloud_high', 10: 'cirrus', 11: 'snow/ice'}

def record(source, inside, present, accepted, affine, crs):
    for label, mask in [('inside',inside), ('present',present), ('accepted',accepted)]:
        if not isinstance(mask,np.ndarray) or mask.dtype != np.bool_ or mask.shape != inside.shape:
            raise ValueError(f'{source}: masca {label} trebuie să fie booleană și să aibă forma {inside.shape}.')
    total = int(inside.sum())
    if total == 0:
        raise ValueError(f'{source}: niciun centru de pixel în oraș; verifică CRS/decupare.')
    if not crs.is_projected or crs.linear_units.lower() not in ('metre','meter','metres','meters'):
        raise ValueError(f'{source}: suprafața aproximativă necesită un CRS proiectat în metri.')
    pixel_area_m2 = abs(affine.a*affine.e-affine.b*affine.d)
    if not np.isfinite(pixel_area_m2) or pixel_area_m2 <= 0:
        raise ValueError(f'{source}: transformare cu suprafață de pixel invalidă.')
    base = inside & present
    valid = base & accepted
    missing = int((inside & ~present).sum())
    excluded = int((base & ~accepted).sum())
    count = int(valid.sum())
    if missing + excluded + count != total:
        raise ValueError(f'{source}: categoriile nu recompun numărul de pixeli din oraș.')
    return dict(sursa=source,
                pixeli_in_oras=total,
                pixeli_lipsa_sau_nefiniti=missing,
                pixeli_exclusi_suplimentar_QA_SCL=excluded,
                pixeli_utilizabili=count,
                procent_utilizabil=100*count/total,
                rezolutie_x_m=float(np.hypot(affine.a,affine.d)),
                rezolutie_y_m=float(np.hypot(affine.b,affine.e)),
                suprafata_utilizabila_aprox_km2=count*pixel_area_m2/1_000_000)

records = []
# Temperatura; QA este o bandă de control, nu o observație termică separată.
a, present, inside, affine, crs = read_city(paths['Landsat_ST_B10'])
qa = categorical_on_grid(paths['Landsat_QA_PIXEL'], a.shape, affine, crs, 65535)
present &= (a != 0) & ((qa & 1) == 0)
records.append(record('Landsat_ST_B10', inside, present, qa_landsat_ok(qa), affine, crs))

for name in ['GHSL_BUILT_H', 'GHSL_BUILT_S', 'GHSL_POP']:
    a, present, inside, affine, crs = read_city(paths[name])
    records.append(record(name, inside, present, np.ones(a.shape, dtype=bool), affine, crs))

scl_path = paths['Sentinel2_SCL_20m']
scl, scl_present, scl_inside, _, _ = read_city(scl_path)
classes, counts = np.unique(scl[scl_inside & scl_present], return_counts=True)
print('Distribuție SCL în oraș (număr pixeli la 20 m):')
for category, count in zip(classes.tolist(), counts.tolist()):
    print(f'  {category:>2} {SCL_NAMES.get(category, "unknown"):<25} {count:>9} ({100*count/counts.sum():.2f}%)')
for band in ['B02_10m', 'B03_10m', 'B04_10m', 'B08_10m', 'B11_20m']:
    name = 'Sentinel2_' + band
    a, present, inside, affine, crs = read_city(paths[name])
    present &= a != 0
    scl_aligned = categorical_on_grid(scl_path, a.shape, affine, crs, 0)
    records.append(record(name, inside, present, scl_ok(scl_aligned), affine, crs))

pixel_coverage = pd.DataFrame(records)
out = P / 'date/acoperire_pixeli.csv'
out.parent.mkdir(parents=True, exist_ok=True)
pixel_coverage.to_csv(out, index=False)
print('\n' + pixel_coverage.to_string(index=False))
print('\nSalvat: date/acoperire_pixeli.csv')
print('Procente pe grile native; intersecția comună se calculează după aliniere.')

print('Suprafața aproximativă = pixeli cu centrul în contur × aria pixelului în CRS; nu este intersecție vectorială exactă sau suprafață geodezică.')


Distribuție SCL în oraș (număr pixeli la 20 m):
   2 dark/topographic_shadow        3109 (0.52%)
   4 vegetation                   249577 (41.79%)
   5 not_vegetated                330380 (55.32%)
   6 water                         13806 (2.31%)
   7 unclassified                    344 (0.06%)

            sursa  pixeli_in_oras  pixeli_lipsa_sau_nefiniti  pixeli_exclusi_suplimentar_QA_SCL  pixeli_utilizabili  procent_utilizabil  rezolutie_x_m  rezolutie_y_m  suprafata_utilizabila_aprox_km2
   Landsat_ST_B10          265436                      24511                                  1              240924           90.765382           30.0           30.0                         216.8316
     GHSL_BUILT_H           23911                          0                                  0               23911          100.000000          100.0          100.0                         239.1100
     GHSL_BUILT_S           23911                          0                                  0            

## Grila comună de 100 m — acoperire pe celulă

Rulare: `.venv/bin/python ruleaza.py grila`. Construim pătrate de 100 m în EPSG:32635, cu originea pe multipli de 100 m. **Păstrăm celulele al căror centru cade în București.** Pătratele de la margine pot include și suprafață din afara conturului; procentul valid privește întregul pătrat, fără decuparea sa la limită.

Transferăm măștile binare prin mediere ponderată (`average`) pe grila comună: **0 este pixel neutilizabil și participă la medie**, nu NoData de ignorat. Procentele sunt aproximări raster ale fracției de suprafață validă. Pentru Sentinel-2 luăm minimul între fracțiile benzilor selectate; acesta este un criteriu per bandă, nu măsurarea intersecției exacte a pixelilor între benzi.

Comparăm praguri exploratorii de 50%, 80% și 100% pentru fiecare sursă și pentru toate împreună. Nu fixăm încă pragul final. Această etapă calculează acoperirea; nu produce încă valorile fizice aliniate.

Export: `date/grila_acoperire.csv`, cu coordonatele centrelor, procentele și eligibilitatea la pragul exploratoriu de 80%.

In [7]:
from math import floor, ceil
from rasterio.warp import transform as transform_coords
from shapely.geometry import shape, mapping, box
from shapely import transform as shape_transform
from rasterio.transform import from_origin

load_native.cache_clear()
categorical_on_grid.cache_clear()
GRID_CRS = 'EPSG:32635'
CELL = 100
city_metric = shape_transform(
    shape(city).segmentize(0.001),
    lambda x, y: tuple(np.asarray(v) for v in transform_coords('EPSG:4326', GRID_CRS, x, y)),
    interleaved=False)
if not city_metric.is_valid:
    raise ValueError('Geometria orașului nu este validă în CRS-ul grilei.')
x0, y0, x1, y1 = city_metric.bounds
left, bottom = floor(x0/CELL)*CELL, floor(y0/CELL)*CELL
right, top = ceil(x1/CELL)*CELL, ceil(y1/CELL)*CELL
width, height = int(round((right-left)/CELL)), int(round((top-bottom)/CELL))
grid_affine = from_origin(left, top, CELL, CELL)
keep = geometry_mask([mapping(city_metric)], out_shape=(height, width),
                     transform=grid_affine, invert=True, all_touched=False)
grid_bounds_geom = mapping(box(left, bottom, right, top))

def valid_fraction(source, kind):
    a, present, affine, crs = load_native(source)
    valid = present.copy()
    if kind == 'landsat':
        valid &= a != 0
        qa = categorical_on_grid(paths['Landsat_QA_PIXEL'], a.shape, affine, crs, 65535)
        valid &= qa_landsat_ok(qa)
    elif kind == 'sentinel':
        valid &= a != 0
        scl_grid = categorical_on_grid(paths['Sentinel2_SCL_20m'], a.shape, affine, crs, 0)
        valid &= scl_ok(scl_grid)
    fraction = np.zeros((height, width), dtype=np.float32)
    reproject(source=valid.astype(np.float32), destination=fraction,
              src_transform=affine, src_crs=crs, src_nodata=None,
              dst_transform=grid_affine, dst_crs=GRID_CRS, dst_nodata=None,
              resampling=Resampling.average)
    return np.clip(fraction, 0, 1)

fractions = {'landsat': valid_fraction('Landsat_ST_B10', 'landsat')}
for key, source in [('ghsl_h','GHSL_BUILT_H'), ('ghsl_s','GHSL_BUILT_S'), ('ghsl_pop','GHSL_POP')]:
    fractions[key] = valid_fraction(source, 'ghsl')
sentinel_fractions = [valid_fraction('Sentinel2_' + b, 'sentinel')
                      for b in ['B02_10m','B03_10m','B04_10m','B08_10m','B11_20m']]
fractions['sentinel_min_benzi'] = np.minimum.reduce(sentinel_fractions)
r, c = np.where(keep)
grid = pd.DataFrame({
    'cell_id': [f'UTM35N_100_{int(left+col*CELL)}_{int(top-(row+1)*CELL)}' for row,col in zip(r,c)],
    'x_centru_m': left+(c+0.5)*CELL,
    'y_centru_m': top-(r+0.5)*CELL,
    'crs': GRID_CRS,
})
for key, fraction in fractions.items():
    grid['valid_pct_' + key] = 100 * fraction[keep]
valid_columns = [k for k in grid if k.startswith('valid_pct_')]
grid['eligibil_toate_80pct'] = (grid[valid_columns] >= 80-0.0001).all(axis=1)
assert grid.cell_id.is_unique and len(grid) > 0
assert np.isfinite(grid[valid_columns].to_numpy()).all()
assert ((grid[valid_columns] >= 0) & (grid[valid_columns] <= 100.0001)).all().all()
grid.to_csv(P / 'date/grila_acoperire.csv', index=False)
print(f'\nGrilă comună: {len(grid)} celule de 100 × 100 m, cu centrul în București.')
for threshold in [50,80,100]:
    passes = grid[valid_columns] >= threshold-0.0001
    print(f'\nPrag {threshold}%:')
    for column in valid_columns:
        print(f'  {column}: {int(passes[column].sum())} celule')
    common = int(passes.all(axis=1).sum())
    print(f'  TOATE sursele: {common} celule ({100*common/len(grid):.2f}%)')
print('\nSalvat: date/grila_acoperire.csv')



Grilă comună: 23892 celule de 100 × 100 m, cu centrul în București.

Prag 50%:
  valid_pct_landsat: 21689 celule
  valid_pct_ghsl_h: 23892 celule
  valid_pct_ghsl_s: 23892 celule
  valid_pct_ghsl_pop: 23892 celule
  valid_pct_sentinel_min_benzi: 23855 celule
  TOATE sursele: 21653 celule (90.63%)

Prag 80%:
  valid_pct_landsat: 21433 celule
  valid_pct_ghsl_h: 23892 celule
  valid_pct_ghsl_s: 23892 celule
  valid_pct_ghsl_pop: 23892 celule
  valid_pct_sentinel_min_benzi: 23661 celule
  TOATE sursele: 21212 celule (88.78%)

Prag 100%:
  valid_pct_landsat: 21158 celule
  valid_pct_ghsl_h: 23892 celule
  valid_pct_ghsl_s: 23892 celule
  valid_pct_ghsl_pop: 23892 celule
  valid_pct_sentinel_min_benzi: 22969 celule
  TOATE sursele: 20285 celule (84.90%)

Salvat: date/grila_acoperire.csv


## Valorile fizice aliniate pe grila de 100 m

Rulare: `.venv/bin/python ruleaza.py aliniere`. Funcțiile sunt încărcate fără reluarea raportului pe pixeli. Grila și regulile de la margine rămân cele documentate mai sus. Nu imputăm golurile.

| Coloană | Calcul / interpretare |
|---|---|
| `lst_c` | Media temperaturilor valide: `DN × multiplicator + offset − 273.15`, factori citiți din MTL. Nu este temperatura aerului. |
| `ndvi` | Media NDVI calculat mai întâi la 10 m din B08 și B04, după conversia reflectanței din XML. Păstrăm SCL 4/5/6, reflectanțe nenegative și sumă > 10⁻⁶; restul sunt lipsă. Nu forțăm valorile prin clipping. |
| `ghsl_agbh_m` | Media AGBH 2018: volum construit raportat la suprafața totală a celulei; nu ANBH (înălțimea raportată exclusiv la construit). |
| `built_m2_est` | BUILT-S 2020: transformăm m² construiți / celulă nativă în fracție construită, mediem pe grila comună și înmulțim cu 10.000 m². |
| `pop_est` | POP 2020: transformăm persoane / celulă nativă în densitate pe m², mediem și înmulțim cu 10.000 m². |

Pentru suprafață și populație presupunem distribuție uniformă în celula GHSL nativă. Transferul de densități prin mediere raster este o aproximare și nu garantează conservarea exactă a totalurilor între Mollweide și UTM; nu pretindem precizie cadastrală sau număr observat de persoane. La lipsuri parțiale, aceste două estimări sunt lăsate goale (nu extrapolăm densitatea validă peste suprafața lipsă).

Mediile temperaturii/NDVI/AGBH folosesc numai valori valide, însoțite de fracția de suport. Pragul **80% este provizoriu** și se marchează în coloană, fără ștergerea rândurilor. NDVI primește propria fracție validă, deoarece conversia poate exclude suplimentar pixeli față de diagnosticul SCL.

Referințe: [USGS — conversia temperaturii](https://www.usgs.gov/landsat-missions/landsat-collection-2-surface-temperature), [Sentinel-2 — conversia reflectanței](https://sentiwiki.copernicus.eu/web/s2-products), [GHSL — definițiile AGBH/ANBH](materiale/GHSL_documentatie.pdf).

Export: `date/aliniate.csv`. Datele observațiilor satelitare și epocile GHSL sunt coloane explicite. În această etapă nu scalăm variabilele, nu rulăm PCA și nu estimăm anomaliile.

In [8]:
import re
import xml.etree.ElementTree as ET

# Parametri citiți din fișierele produselor, fără valori implicite.
mtls = list((P / 'date/landsat').glob('*_MTL.txt'))
if len(mtls) != 1:
    raise ValueError('Este necesar exact un MTL Landsat.')
mtl = mtls[0].read_text()
def mtl_number(key):
    match = re.search(r'^\s*' + re.escape(key) + r'\s*=\s*([-+0-9.eE]+)\s*$', mtl, re.M)
    if match is None:
        raise ValueError('Factor absent în MTL: ' + key)
    return float(match.group(1))
mult = mtl_number('TEMPERATURE_MULT_BAND_ST_B10')
add = mtl_number('TEMPERATURE_ADD_BAND_ST_B10')
date_match = re.search(r'^\s*DATE_ACQUIRED\s*=\s*(\d{4}-\d{2}-\d{2})', mtl, re.M)
if date_match is None:
    raise ValueError('Lipsește data Landsat.')
xmls = list((P / 'date/sentinel2').glob('*.SAFE/MTD_MSIL2A.xml'))
if len(xmls) != 1:
    raise ValueError('Este necesar exact un produs Sentinel-2 pentru această rulare.')
if 'T35TMK' not in xmls[0].parent.name:
    raise ValueError('Le pipeline curent este configurat pentru dala T35TMK.')
xml = ET.parse(xmls[0])
def elements(tag):
    return [e for e in xml.iter() if e.tag.split('}')[-1] == tag]
quant = float(elements('BOA_QUANTIFICATION_VALUE')[0].text)
if quant <= 0:
    raise ValueError('Factor de cuantificare Sentinel invalid.')
band_ids = {e.attrib['physicalBand']: e.attrib['bandId'] for e in elements('Spectral_Information')}
offsets = {e.attrib['band_id']: float(e.text) for e in elements('BOA_ADD_OFFSET')}
offset_red, offset_nir = offsets[band_ids['B4']], offsets[band_ids['B8']]
s2_date = elements('PRODUCT_START_TIME')[0].text[:10]
print(f'Factori Landsat: {mult}, {add}; Sentinel: Q={quant}, B04={offset_red}, B08={offset_nir}')

def aggregate_native(values, valid, affine, crs, known_support=None):
    valid = valid & np.isfinite(values)
    source = np.where(valid, values, np.nan)
    average = np.full((height, width), np.nan, dtype=np.float64)
    support = np.zeros((height, width), dtype=np.float32)
    reproject(source=source, destination=average,
              src_transform=affine, src_crs=crs, src_nodata=np.nan,
              dst_transform=grid_affine, dst_crs=GRID_CRS, dst_nodata=np.nan,
              resampling=Resampling.average)
    if known_support is None:
        reproject(source=valid.astype(np.float32), destination=support,
                  src_transform=affine, src_crs=crs, src_nodata=None,
                  dst_transform=grid_affine, dst_crs=GRID_CRS, dst_nodata=None,
                  resampling=Resampling.average)
    else:
        support[:] = known_support
    return average[keep], np.clip(support[keep]*100, 0, 100)

aligned = grid.copy()
a, valid, affine, crs = load_native('Landsat_ST_B10')
qa = categorical_on_grid(paths['Landsat_QA_PIXEL'], a.shape, affine, crs, 65535)
valid = valid.copy()
valid &= (a != 0) & qa_landsat_ok(qa)
aligned['lst_c'], aligned['support_lst_pct'] = aggregate_native(a*mult+add-273.15, valid, affine, crs, fractions['landsat'])

red_dn, red_valid, affine, crs = load_native('Sentinel2_B04_10m')
nir_dn, nir_valid, nir_affine, nir_crs = load_native('Sentinel2_B08_10m')
if red_dn.shape != nir_dn.shape or affine != nir_affine or crs != nir_crs:
    raise ValueError('B04 și B08 nu sunt pe aceeași grilă nativă.')
red, nir = (red_dn+offset_red)/quant, (nir_dn+offset_nir)/quant
scl_grid = categorical_on_grid(paths['Sentinel2_SCL_20m'], red_dn.shape, affine, crs, 0)
valid = (red_valid & nir_valid & (red_dn != 0) & (nir_dn != 0) & scl_ok(scl_grid)
         & (red >= 0) & (nir >= 0) & ((nir+red) > 1e-6))
ndvi_native = np.full(red.shape, np.nan)
np.divide(nir-red, nir+red, out=ndvi_native, where=valid)
aligned['ndvi'], aligned['support_ndvi_pct'] = aggregate_native(ndvi_native, valid, affine, crs)

for source, column, density in [
        ('GHSL_BUILT_H','ghsl_agbh_m',False),
        ('GHSL_BUILT_S','built_m2_est',True),
        ('GHSL_POP','pop_est',True)]:
    a, valid, affine, crs = load_native(source)
    if density:
        if crs.to_string() != 'ESRI:54009':
            raise ValueError('La densități presupunem celule native Mollweide în metri.')
        native_area = abs(affine.a*affine.e-affine.b*affine.d)
        a = a/native_area
    support_key = {'GHSL_BUILT_H':'ghsl_h','GHSL_BUILT_S':'ghsl_s','GHSL_POP':'ghsl_pop'}[source]
    means, support = aggregate_native(a, valid, affine, crs, fractions[support_key])
    if density:
        means *= CELL*CELL
        means[support < 100-0.0001] = np.nan
    aligned[column] = means
    aligned['support_' + column + '_pct'] = support

aligned['data_landsat'] = date_match.group(1)
aligned['data_sentinel2'] = s2_date
aligned['epoca_ghsl_h'] = 2018
aligned['epoca_ghsl_s'] = 2020
aligned['epoca_ghsl_pop'] = 2020
physical_columns = ['lst_c','ndvi','ghsl_agbh_m','built_m2_est','pop_est']
support_columns = [c for c in aligned if c.startswith('support_')]
aligned['eligibil_valori_80pct'] = (
    aligned['eligibil_toate_80pct'] & aligned[physical_columns].notna().all(axis=1)
    & (aligned[support_columns] >= 80-0.0001).all(axis=1))
if not aligned.cell_id.is_unique or len(aligned) != len(grid):
    raise ValueError('Chei duplicate sau număr de celule schimbat.')
if not aligned['ndvi'].dropna().between(-1.000001,1.000001).all():
    raise ValueError('NDVI în afara intervalului fizic.')
for c in ['ghsl_agbh_m','built_m2_est','pop_est']:
    if (aligned[c].dropna() < 0).any():
        raise ValueError('Valori negative neașteptate: ' + c)
if (aligned['built_m2_est'].dropna() > CELL*CELL+0.01).any():
    raise ValueError('Suprafață construită mai mare decât celula.')
if np.isinf(aligned[physical_columns].to_numpy()).any():
    raise ValueError('Valori infinite în tabel.')
aligned.to_csv(P / 'date/aliniate.csv', index=False)
print(f'\nTabel: {len(aligned)} celule păstrate; {int(aligned.eligibil_valori_80pct.sum())} eligibile la pragul provizoriu 80%.')
print('\nRezumat pentru celulele eligibile:')
print(aligned.loc[aligned.eligibil_valori_80pct,physical_columns].describe().to_string())
print('\nValori lipsă în întregul tabel:')
print(aligned[physical_columns].isna().sum().to_string())
print('\nSalvat: date/aliniate.csv. AGBH nu este ANBH; populația și construitul sunt estimări transferate prin densitate.')


Factori Landsat: 0.00341802, 149.0; Sentinel: Q=10000.0, B04=-1000.0, B08=-1000.0

Tabel: 23892 celule păstrate; 21212 eligibile la pragul provizoriu 80%.

Rezumat pentru celulele eligibile:
              lst_c          ndvi   ghsl_agbh_m  built_m2_est       pop_est
count  21212.000000  21212.000000  21212.000000  21212.000000  21212.000000
mean      40.122114      0.402326      3.123003   2471.414352     81.427668
std        3.708268      0.186448      2.794783   1662.247922     85.643384
min       27.952813     -0.238232      0.000000      0.000000      0.000000
25%       38.280087      0.266646      0.385538    809.532442      6.973370
50%       40.620802      0.383268      2.847648   2850.029184     56.738191
75%       42.518654      0.520707      4.993277   3779.573631    127.068822
max       54.663562      0.882956     19.081136   9694.193706    590.260455

Valori lipsă în întregul tabel:
lst_c           1723
ndvi               0
ghsl_agbh_m        0
built_m2_est       0
pop_est 

## Hărți de verificare — temperatură și excluderi

Rulare: `.venv/bin/python ruleaza.py harti`. Citește numai `date/aliniate.csv` și conturul, fără reluarea rasterelor. Produce `date/verificare_harti.png`.

Stânga: temperatura suprafeței pentru celulele eligibile la pragul provizoriu 80%; maximul **în acest subset** este marcat. Dreapta: toate celulele orașului, clasificate după eligibilitate. Motivele sunt exclusive în ordinea: temperatură lipsă → suport termic sub 80% → suport optic/NDVI sub 80% → alte condiții. O celulă poate avea și alte probleme decât motivul prioritar afișat.

Afișăm și coordonatele, variabilele și suportul maximului, plus vecinii aflați la maximum 200 m. Aceste controale ajută inspecția, dar nu validează automat maximul. Valorile și golurile provin dintr-o singură observație termică din 17.08.2026, nu din media verii.

In [2]:
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mede-matplotlib')
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from rasterio.warp import transform as coords_transform
from shapely.geometry import shape
from shapely import transform as geom_transform

root = Path.cwd()
d = pd.read_csv(root / 'date/aliniate.csv')
def boolean_column(name):
    text = d[name].astype(str).str.lower()
    if not text.isin(['true','false']).all():
        raise ValueError('Indicator boolean invalid: ' + name)
    return text.eq('true')
eligible = boolean_column('eligibil_valori_80pct')
if not eligible.any():
    raise ValueError('Nu există celule eligibile.')
if set(d.crs) != {'EPSG:32635'}:
    raise ValueError('Harta așteaptă UTM 35N.')
status = np.full(len(d), 4, dtype=int)
missing = d.lst_c.isna()
thermal = ~missing & (d.support_lst_pct < 80-0.0001)
optical = ~missing & ~thermal & ((d.valid_pct_sentinel_min_benzi < 80-0.0001)
                                | (d.support_ndvi_pct < 80-0.0001) | d.ndvi.isna())
status[missing] = 1
status[thermal] = 2
status[optical] = 3
status[eligible] = 0
labels = ['Eligibilă ≥80%', 'Temperatură lipsă', 'Suport termic <80%',
          'Suport optic/NDVI <80%', 'Alte condiții neîndeplinite']
colors = ['#dce8dc','#252525','#e8903b','#8470b2','#bc4444']
xs = np.sort(d.x_centru_m.unique())
ys = np.sort(d.y_centru_m.unique())[::-1]
if not np.allclose(np.diff(xs),100) or not np.allclose(np.diff(ys),-100):
    raise ValueError('Centrele nu descriu grila regulată de 100 m.')
ci = np.rint((d.x_centru_m-xs[0])/100).astype(int).to_numpy()
ri = np.rint((ys[0]-d.y_centru_m)/100).astype(int).to_numpy()
if d.duplicated(['x_centru_m','y_centru_m']).any():
    raise ValueError('Centre de celule duplicate.')
temp_image = np.full((len(ys),len(xs)),np.nan)
status_image = np.full(temp_image.shape,np.nan)
temp_image[ri[eligible],ci[eligible]] = d.loc[eligible,'lst_c']
status_image[ri,ci] = status
extent = [(xs[0]-50)/1000,(xs[-1]+50)/1000,(ys[-1]-50)/1000,(ys[0]+50)/1000]
features = json.loads((root/'date/bucuresti.geojson').read_text())['features']
geom = next(f['geometry'] for f in features if f.get('properties',{}).get('@id')=='relation/377733')
outline = geom_transform(shape(geom),
    lambda x,y: tuple(np.asarray(v) for v in coords_transform('EPSG:4326','EPSG:32635',x,y)),
    interleaved=False)
maximum = d.loc[d.loc[eligible,'lst_c'].idxmax()]
fig, axes = plt.subplots(1,2,figsize=(13,7),layout='constrained')
im = axes[0].imshow(temp_image, origin='upper',extent=extent,cmap='inferno',interpolation='nearest')
fig.colorbar(im,ax=axes[0],shrink=.65,label='Temperatura suprafeței (°C)')
axes[0].scatter(maximum.x_centru_m/1000,maximum.y_centru_m/1000,marker='x',s=70,
                linewidths=1.6,color='#00d5ff',label=f'Maxim eligibil: {maximum.lst_c:.2f} °C')
axes[0].legend(loc='lower left',fontsize=8)
axes[0].set_title('Temperatura celulelor eligibile · 17.08.2026')
axes[1].imshow(status_image,origin='upper',extent=extent,cmap=ListedColormap(colors),
               norm=BoundaryNorm(np.arange(-.5,5.5),5),interpolation='nearest')
legend = [Patch(facecolor=colors[i],label=f'{label}: {int((status==i).sum()):,}') for i,label in enumerate(labels)]
axes[1].legend(handles=legend,loc='lower left',fontsize=8)
axes[1].set_title('Eligibilitate și motiv prioritar de excludere')
for ax in axes:
    for poly in ([outline] if outline.geom_type=='Polygon' else outline.geoms):
        xy=np.asarray(poly.exterior.coords)/1000
        ax.plot(xy[:,0],xy[:,1],color='#388fbb',linewidth=.6)
        for ring in poly.interiors:
            xy=np.asarray(ring.coords)/1000
            ax.plot(xy[:,0],xy[:,1],color='#388fbb',linewidth=.6)
    ax.set_xlabel('Est UTM 35N (km)')
    ax.set_ylabel('Nord UTM 35N (km)')
    ax.set_aspect('equal')
    ax.ticklabel_format(style='plain',useOffset=False)
fig.suptitle('București · grilă 100 m · verificare înainte de analiză',fontsize=13)
fig.supxlabel('Landsat 8 C2 L2: USGS/NASA · Contains modified Copernicus Sentinel data 2026 (acces: CDSE)\nGHSL © European Union / JRC, CC BY 4.0: Pesaresi et al. (2024); Pesaresi & Politis (2023); Schiavina et al. (2023)\nContur © OpenStreetMap contributors, ODbL · openstreetmap.org/copyright · Referințe complete: datacard_U13.pdf',fontsize=7)
path=root/'date/verificare_harti.png'
fig.savefig(path,dpi=180)
plt.close(fig)
lon,lat=coords_transform('EPSG:32635','EPSG:4326',[maximum.x_centru_m],[maximum.y_centru_m])
print('Maxim între celulele eligibile:')
print(maximum[['cell_id','lst_c','support_lst_pct','ndvi','support_ndvi_pct',
               'ghsl_agbh_m','built_m2_est','pop_est']].to_string())
print(f'Centru WGS84: latitudine {lat[0]:.6f}, longitudine {lon[0]:.6f}')
distance=np.hypot(d.x_centru_m-maximum.x_centru_m,d.y_centru_m-maximum.y_centru_m)
neighbors=d.loc[(distance>0)&(distance<=200)&eligible,'lst_c']
print(f'Vecini eligibili la maximum 200 m: {len(neighbors)}; '
      f'min={neighbors.min():.2f}, mediană={neighbors.median():.2f}, max={neighbors.max():.2f} °C')
print('\nMotive prioritare (nu se adună dublu):')
for i,label in enumerate(labels):
    print(f'{label}: {int((status==i).sum())}')
assert len(d)==sum(int((status==i).sum()) for i in range(5))
print(f'\nHartă salvată: {path.relative_to(root)}')

Maxim între celulele eligibile:
cell_id             UTM35N_100_430400_4913800
lst_c                               54.663562
support_lst_pct                         100.0
ndvi                                 0.136129
support_ndvi_pct                        100.0
ghsl_agbh_m                          3.074637
built_m2_est                      2951.277399
pop_est                             34.725645
Centru WGS84: latitudine 44.374593, longitudine 26.127002
Vecini eligibili la maximum 200 m: 12; min=41.47, mediană=48.41, max=51.87 °C

Motive prioritare (nu se adună dublu):
Eligibilă ≥80%: 21212
Temperatură lipsă: 1723
Suport termic <80%: 736
Suport optic/NDVI <80%: 221
Alte condiții neîndeplinite: 0

Hartă salvată: date/verificare_harti.png


## Diagnostic termic — maxim și goluri

Rulare: `.venv/bin/python ruleaza.py diagnostic`, fără regenerarea hărților sau alinierii. Citește tabelul și ferestrele Landsat din jurul orașului și maximului. Afișează maximul eligibil, vecinii, pixelii termici nativi și incertitudinea ST_QA. Separat, numără marcajele QA peste pixelii fără temperatură. Marcajele se pot suprapune; numerele lor nu se adună.

ST_QA este o bandă distinctă de incertitudine; afișăm valorile stocate, fără a presupune aici factorul de conversie. Nu identificăm automat cauza golurilor doar din distribuția lor sau din QA. Un QA fără marcaj de nor nu demonstrează că golul este lipsit de orice altă problemă.

Pentru compararea suprafețelor, afișăm și medianele NDVI/construitului pentru celulele fără temperatură și cele eligibile. Diferențele sunt descriptive, nu dovezi cauzale.

In [10]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_window, geometry_mask
from rasterio.windows import from_bounds
from rasterio.warp import transform as tr, transform_geom

base = Path.cwd()
table = pd.read_csv(base/'date/aliniate.csv')
flag = table.eligibil_valori_80pct.astype(str).str.lower()
if not flag.isin(['true','false']).all():
    raise ValueError('Indicator de eligibilitate invalid.')
valid_rows = flag.eq('true')
maximum = table.loc[table.loc[valid_rows,'lst_c'].idxmax()]
lon,lat=tr('EPSG:32635','EPSG:4326',[maximum.x_centru_m],[maximum.y_centru_m])
print('MAXIMUL ELIGIBIL')
print(maximum[['cell_id','lst_c','support_lst_pct','ndvi','support_ndvi_pct',
               'ghsl_agbh_m','built_m2_est','pop_est']].to_string())
print(f'Centru: lat={lat[0]:.6f}, lon={lon[0]:.6f}')
distance=np.hypot(table.x_centru_m-maximum.x_centru_m,table.y_centru_m-maximum.y_centru_m)
neighbors=table.loc[(distance>0)&(distance<=200)&valid_rows,'lst_c']
print(f'Vecini la ≤200 m: n={len(neighbors)}, min={neighbors.min():.3f}, '
      f'mediană={neighbors.median():.3f}, max={neighbors.max():.3f} °C')

files=dict(files)
mtls=list((base/'date/landsat').glob('*_MTL.txt'))
if len(mtls)!=1:raise ValueError('Este necesar exact un MTL.')
mtl=mtls[0].read_text()
def factor(key):
    m=re.search(r'^\s*'+re.escape(key)+r'\s*=\s*([-+0-9.eE]+)\s*$',mtl,re.M)
    if not m:raise ValueError('Factor absent: '+key)
    return float(m.group(1))
mul=factor('TEMPERATURE_MULT_BAND_ST_B10');add=factor('TEMPERATURE_ADD_BAND_ST_B10')
features=json.loads((base/'date/bucuresti.geojson').read_text())['features']
city=next(f['geometry'] for f in features if f.get('properties',{}).get('@id')=='relation/377733')
with rasterio.open(files['Landsat_ST_B10']) as st, rasterio.open(files['Landsat_QA_PIXEL']) as qa, rasterio.open(files['Landsat_ST_QA']) as uncertainty:
    for ds in [qa,uncertainty]:
        if (ds.crs!=st.crs or ds.transform!=st.transform or ds.shape!=st.shape):
            raise ValueError('Benzile Landsat nu sunt pe aceeași grilă.')
    city_geom=transform_geom('EPSG:4326',st.crs,city)
    win=geometry_window(st,[city_geom])
    raw=st.read(1,window=win,masked=True)
    q=qa.read(1,window=win)
    inside=geometry_mask([city_geom],out_shape=raw.shape,transform=st.window_transform(win),invert=True)
    missing=inside & (np.ma.getmaskarray(raw) | (raw.data==0) | ~np.isfinite(raw.data))
    print(f'\nGOLURI TERMICE NATIVE ÎN ORAȘ: {int(missing.sum())}')
    for bit,label in [(0,'fill'),(1,'dilated cloud'),(2,'cirrus'),(3,'cloud'),(4,'cloud shadow'),(5,'snow'),(7,'water')]:
        print(f'  cu marcaj {label}: {int((missing & ((q & (1<<bit))!=0)).sum())}')
    print('  fără niciun marcaj QA 0–5:',int((missing & ((q & 63)==0)).sum()))
    x,y=tr('EPSG:32635',st.crs,[maximum.x_centru_m],[maximum.y_centru_m])
    win=from_bounds(x[0]-200,y[0]-200,x[0]+200,y[0]+200,st.transform).round_offsets().round_lengths()
    native=st.read(1,window=win,masked=True)
    native_qa=qa.read(1,window=win)
    good=(~np.ma.getmaskarray(native)) & (native.data!=0) & ((native_qa & 63)==0)
    temperatures=native.data[good].astype(float)*mul+add-273.15
    print('\nFEREASTRĂ NATIVĂ APROXIMATIV 400 × 400 m ÎN JURUL MAXIMULUI')
    if temperatures.size:
        print(f'Temperaturi acceptate QA: n={temperatures.size}; min={temperatures.min():.3f}, '
              f'mediană={np.median(temperatures):.3f}, max={temperatures.max():.3f} °C')
    u=uncertainty.read(1,window=win,masked=True)
    u_valid=good & ~np.ma.getmaskarray(u)
    if u_valid.any():
        print('ST_QA stocat, min/mediană/max:',float(u.data[u_valid].min()),
              float(np.median(u.data[u_valid])),float(u.data[u_valid].max()))

print('\nCOMPARAȚIE DESCRIPTIVĂ A CELULELOR (mediană)')
for name, subset in [('Fără temperatură',table.lst_c.isna()),('Eligibile',valid_rows)]:
    print(name, 'n=',int(subset.sum()))
    print(table.loc[subset,['ndvi','built_m2_est','ghsl_agbh_m']].median().to_string())
print('\nCauza golurilor și validitatea maximului nu sunt demonstrate de acest diagnostic.')


MAXIMUL ELIGIBIL
cell_id             UTM35N_100_430400_4913800
lst_c                               54.663562
support_lst_pct                         100.0
ndvi                                 0.136129
support_ndvi_pct                        100.0
ghsl_agbh_m                          3.074637
built_m2_est                      2951.277399
pop_est                             34.725645
Centru: lat=44.374593, lon=26.127002
Vecini la ≤200 m: n=12, min=41.468, mediană=48.414, max=51.868 °C

GOLURI TERMICE NATIVE ÎN ORAȘ: 24511
  cu marcaj fill: 0
  cu marcaj dilated cloud: 0
  cu marcaj cirrus: 0
  cu marcaj cloud: 0
  cu marcaj cloud shadow: 0
  cu marcaj snow: 0
  cu marcaj water: 0
  fără niciun marcaj QA 0–5: 24511

FEREASTRĂ NATIVĂ APROXIMATIV 400 × 400 m ÎN JURUL MAXIMULUI
Temperaturi acceptate QA: n=169; min=40.527, mediană=47.759, max=57.713 °C
ST_QA stocat, min/mediană/max: 205.0 209.0 222.0

COMPARAȚIE DESCRIPTIVĂ A CELULELOR (mediană)
Fără temperatură n= 1723
ndvi               0.4